# Data Loading

In [1]:
import pandas as pd
import re
import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [42]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
reviews_df.head()

,app_id,app_name,review_text,review_score,review_votes
0,10,Counter-Strike,Ruined my life.,1,0
1,10,Counter-Strike,This will be more of a ''my experience with th...,1,1
2,10,Counter-Strike,This game saved my virginity.,1,0
3,10,Counter-Strike,• Do you like original games? • Do you like ga...,1,0
4,10,Counter-Strike,"Easy to learn, hard to master.",1,1


In [ ]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

{'Utilities', 'Movie', 'Action', 'Massively Multiplayer', 'Episodic', 'Sports', 'Design & Illustration', 'Casual', 'Free To Play', 'Video Production', 'Audio Production', 'Photo Editing', 'Violent', 'Documentary', 'Strategy', 'Web Publishing', 'Nudity', 'Tutorial', 'Accounting', 'Adventure', '360 Video', 'Sexual Content', 'Racing', 'Education', 'Early Access', 'Game Development', 'Animation & Modeling', 'Simulation', 'RPG', 'Short', 'Software Training', 'Indie', 'Gore', 'nan'}
34
{'Utilities', 'Movie', 'Action', 'Massively Multiplayer', 'Episodic', 'Sports', 'Design & Illustration', 'Casual', 'Free To Play', 'Video Production', 'Audio Production', 'Photo Editing', 'Violent', 'Documentary', 'Strategy', 'Web Publishing', 'Nudity', 'Tutorial', 'Accounting', 'Adventure', '360 Video', 'Sexual Content', 'Racing', 'Education', 'Early Access', 'Game Development', 'Animation & Modeling', 'Simulation', 'RPG', 'Short', 'Software Training', 'Indie', 'Gore', 'nan'}
34
{'Souls-like', 'Turn-Based Com

In [29]:

narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
1108640,OMNIMUS,"Sep 25, 2019",0 - 20000,0,0,0.99,0,1,Omnimus is an interactive visual novel in an u...,"['English', 'Russian']",...,0,0,0,Angry games,Total,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Ficti...",https://shared.akamai.steamstatic.com/store_it...,NaN
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",20000 - 50000,0,0,12.49,0,3,■Defy Death by Rewinding Time! Any hit you tak...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,"Gemdrops, Inc.","Gemdrops, Inc.","Single-player,Steam Achievements,Full controll...",Action,"Action-Adventure,Pixel Graphics,Time Manipulat...",https://shared.akamai.steamstatic.com/store_it...,NaN
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0 - 20000,0,0,9.99,0,0,A stylish &amp; exciting action game! Journey ...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,SERIALGAMES Inc.,Gotcha Gotcha Games,"Single-player,Full controller support,Family S...","Action,Indie","Action,Arcade,Character Action Game,2D Platfor...",https://shared.akamai.steamstatic.com/store_it...,NaN
3098390,Spookstopia: Atomic Division,"Jul 31, 2024",0 - 20000,0,0,2.99,0,0,What is possible between the dimensions of Hel...,"['English', 'Russian']",...,0,0,0,TexFrendza,TexFrendza,"Single-player,Family Sharing","Indie,Early Access","Adventure,Platformer,2D Platformer,2D,Mystery,...",https://shared.akamai.steamstatic.com/store_it...,NaN
3215680,Who is Paul,"Oct 21, 2024",0 - 20000,0,0,0.89,70,0,WARNING: The game's content includes sudden lo...,"['English', 'French', 'Italian', 'German', 'Sp...",...,0,0,0,Atlas Sf,Atlas Sf,"Single-player,Steam Achievements,Family Sharing","Action,Adventure,Indie","Action,Adventure,FPS,Dark,Story Rich,Stealth,A...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [47]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

filtered_reviews

,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,"First of all, this is no game but a virtual in...",1,1
46178,104600,Portal 2 - The Final Hours,The Final Hours is not a game (though can't yo...,1,1
46179,104600,Portal 2 - The Final Hours,Get this if you are a geek about how games are...,1,1
46180,104600,Portal 2 - The Final Hours,IT'S NOT A GAME! But it's a very interesting r...,1,1
46181,104600,Portal 2 - The Final Hours,Everything you would want to know about Portal...,1,1
...,...,...,...,...,...
6404109,9880,Champions Online,the gane starts to get boreing when you reatch...,-1,0
6404110,9880,Champions Online,Best character creator i've seen. EDIT: They ...,-1,0
6404111,9880,Champions Online,Region banned!! why did u show me this game to...,-1,0
6404112,9880,Champions Online,why the ♥♥♥♥,-1,0
